<a href="https://colab.research.google.com/github/BistroP/CS182-final-project/blob/main/colab_cifar_hypothesis1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Quasi-norm Muon: run Hypothesis 1 on CIFAR-10 (Colab or local GPU)

ViT (dim 256, depth 6, ~3.2M params) on CIFAR-10: Muon vs. Kaon vs. Freon (c = 2/3, 3/4) vs. SGD / TruncatedSGD.

Works on a **hosted Colab GPU** (Runtime → Change runtime type → GPU; A100 if you can get it, T4 is much slower)
or a **local runtime** with your own NVIDIA GPU (see the setup cell). Run top to bottom. Every sweep is
**resumable**: if the runtime disconnects, re-run the setup cells and the sweep cell again and it skips finished
runs (on hosted Colab, results are saved to Drive if you mount it).

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print("torch", torch.__version__, "cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    cap = "sm_%d%d" % torch.cuda.get_device_capability()
    if cap not in torch.cuda.get_arch_list():   # e.g. RTX 50xx (sm_120) needs a CUDA 12.8+ build of torch
        print(f"WARNING: this torch build has no kernels for {cap}; install a newer one, e.g.\n"
              "  pip install --upgrade torch torchvision --index-url https://download.pytorch.org/whl/cu128")

name, memory.total [MiB]
NVIDIA GeForce RTX 5080, 16303 MiB
torch 2.14.1+cu130 cuda True


## 1. Setup
- **Hosted Colab:** clones the repo to `/content/repo`; results go to Drive if `USE_DRIVE`.
- **Local runtime:** set `LOCAL_REPO` to your checkout of this repo (it is cloned there if missing); results go to
  `<LOCAL_REPO>/results`. Drive can't be mounted on a local runtime. Install deps once in the Jupyter environment:
  `pip install -r requirements.txt` (and the CUDA 12.8 torch above for RTX 50xx cards).

In [2]:
import os, sys
IN_COLAB = os.path.exists("/var/colab/hostname")   # hosted Colab only; False on a local runtime
USE_DRIVE = True   # hosted Colab: set False to keep results only on the (ephemeral) Colab disk
LOCAL_REPO = os.path.expanduser("~/CS182-final-project")   # local runtime: path to your checkout
REPO = "/content/repo" if IN_COLAB else LOCAL_REPO
RESULTS = "/content/results" if IN_COLAB else os.path.join(REPO, "results")
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RESULTS = "/content/drive/MyDrive/cs182_results"
print("colab" if IN_COLAB else "local runtime", "| repo:", REPO, "| python:", sys.executable)

local runtime | repo: /root/CS182-final-project | python: /usr/bin/python3


In [3]:
if not os.path.isdir(REPO):
    !git clone https://github.com/BistroP/CS182-final-project.git {REPO}
else:
    !git -C {REPO} pull
%cd {REPO}
os.makedirs(RESULTS, exist_ok=True)
print("results dir:", RESULTS)
!{sys.executable} -m pytest -q tests

Cloning into '/root/CS182-final-project'...
remote: Enumerating objects: 124, done.
remote: Counting objects: 100% (124/124), done.
remote: Compressing objects: 100% (84/84), done.
remote: Total 124 (delta 54), reused 102 (delta 37), pack-reused 0 (from 0)
Receiving objects: 100% (124/124), 71.33 KiB | 1.32 MiB/s, done.
Resolving deltas: 100% (54/54), done.
/root/CS182-final-project
results dir: /root/CS182-final-project/results
.................................................                        [100%]
49 passed in 8.25s


In [4]:
# Downloads CIFAR-10 (~170 MB) to data/cifar10 via torchvision, then a 1-epoch smoke run on it
!{sys.executable} train_cifar.py --optimizer muon --epochs 1 --out_dir runs/cifar/smoke

100% 170M/170M [01:07<00:00, 2.54MB/s]
vit muon c=0.667 seed=42 device=cuda params=3.18M spectral=24 steps/epoch=97
epoch   0 train nan test 2.3210 acc 0.0802 (1s)
epoch   1 train 1.7689 test 1.4064 acc 0.4937 (4s)


## 2. Benchmark (≈ 3–5 min) — do this first
Times one CIFAR epoch (97 steps at batch 512, + test eval) per optimizer. Look at the **freon svd vs eigh**
rows and pick the faster one for `FREON_IMPL` below. The last column extrapolates the whole stage-1 + stage-2 cost.

In [ ]:
import time
from train_cifar import CifarConfig, run

BENCH = {"muon": {}, "kaon": {}, "freon svd": dict(optimizer="freon", freon_impl="svd"),
         "freon eigh": dict(optimizer="freon", freon_impl="eigh"), "sgd": dict(momentum=0.9),
         "truncsgd": dict(momentum=0.9)}
EPOCHS_PER_OPT = 16 * 30 + 3 * 30   # stage 1 (16-point grid x 30 epochs) + stage 2 (3 seeds x 30 epochs)
sec_per_epoch = {}
for name, kw in BENCH.items():
    kw = {"optimizer": name.split()[0], **kw}
    res = run(CifarConfig(**kw, epochs=2, out_dir=f"runs/cifar/bench_{name.replace(' ', '_')}"), verbose=False)
    sec_per_epoch[name] = res["log"][-1]["time"] - res["log"][-2]["time"]   # last epoch only (skips warm-up)
    print(f"{name:12s} {sec_per_epoch[name]:6.2f} s/epoch   ~{sec_per_epoch[name] * EPOCHS_PER_OPT / 3600:5.1f} h for H1")

muon           2.53 s/epoch   ~  0.4 h for H1
kaon           2.54 s/epoch   ~  0.4 h for H1
freon svd     26.65 s/epoch   ~  4.2 h for H1
freon eigh    12.49 s/epoch   ~  2.0 h for H1
sgd            2.26 s/epoch   ~  0.4 h for H1
truncsgd      31.02 s/epoch   ~  4.9 h for H1


In [6]:
FREON_IMPL = "eigh"   # <- set to "svd" if the benchmark shows svd is faster

## 3. Stage 1 — tune learning rates (seed 42, 30 epochs per run)
16 `(matrix_lr, base_lr)` points per spectral optimizer (15 for SGD / TruncatedSGD, which have their own larger-lr grid).
That's 16×4 + 15×2 = 94 runs. Kaon is Muon-cost; Freon cost depends on the benchmark.

To save compute you can tune on shorter runs (e.g. `TUNE_EPOCHS = 15`), at the cost that the best lr can shift a
little with run length — same caveat as the Shakespeare protocol.

In [7]:
import sys
OPTS = "muon kaon freon:0.6667 freon:0.75 sgd truncsgd:0.05"
TUNE_EPOCHS = 30
!{sys.executable} sweep_cifar.py tune --opts {OPTS} --freon_impl {FREON_IMPL} --set epochs={TUNE_EPOCHS} --out {RESULTS}/cifar_tune.jsonl

{'mode': 'tune', 'tag': 'sgd', 'arch': 'vit', 'matrix_lr': 0.5, 'base_lr': 0.002, 'seed': 42, 'rho': None, 'c': None, 'final_test_loss': 0.6001803771972656, 'final_test_acc': 0.7995, 'best_test_acc': 0.8038, 'diverged': False}
{'mode': 'tune', 'tag': 'sgd', 'arch': 'vit', 'matrix_lr': 1.0, 'base_lr': 0.0005, 'seed': 42, 'rho': None, 'c': None, 'final_test_loss': 0.5939427612304687, 'final_test_acc': 0.7987, 'best_test_acc': 0.8042, 'diverged': False}
{'mode': 'tune', 'tag': 'sgd', 'arch': 'vit', 'matrix_lr': 1.0, 'base_lr': 0.001, 'seed': 42, 'rho': None, 'c': None, 'final_test_loss': 0.5994597473144532, 'final_test_acc': 0.7968, 'best_test_acc': 0.7993, 'diverged': False}
{'mode': 'tune', 'tag': 'sgd', 'arch': 'vit', 'matrix_lr': 1.0, 'base_lr': 0.002, 'seed': 42, 'rho': None, 'c': None, 'final_test_loss': 0.5997866516113282, 'final_test_acc': 0.7998, 'best_test_acc': 0.8033, 'diverged': False}
{'mode': 'tune', 'tag': 'truncsgd:0.05', 'arch': 'vit', 'matrix_lr': 0.05, 'base_lr': 0.000

In [8]:
# Best spec:matrix_lr:base_lr per optimizer (by final test acc); warns if the best lr is on the grid edge
!{sys.executable} analyze_cifar.py tune {RESULTS}/cifar_tune.jsonl

optimiser          matrix_lr   base_lr  test acc  test loss
freon:0.6667            0.01     0.001    0.8490     0.6558
freon:0.75              0.01    0.0005    0.8454     0.6523
  ! freon:0.75: best base_lr is on the grid edge, widen the grid
kaon                    0.01    0.0005    0.8476     0.6688
  ! kaon: best base_lr is on the grid edge, widen the grid
muon                    0.01     0.002    0.8480     0.6905
sgd                      0.5     0.001    0.8006     0.6108
truncsgd:0.05            0.5     0.002    0.8102     0.6504
  ! truncsgd:0.05: best base_lr is on the grid edge, widen the grid

scale command:
  python sweep_cifar.py scale --opts freon:0.6667:0.01:0.001 freon:0.75:0.01:0.0005 kaon:0.01:0.0005 muon:0.01:0.002 sgd:0.5:0.001 truncsgd:0.05:0.5:0.002 --out results/cifar_scale.jsonl


## 4. Stage 2 — final runs, 3 seeds × 30 epochs at each optimizer's best lrs
Paste the `spec:matrix_lr:base_lr` strings from the `scale command` printed above into `FINAL_SPECS`.
(`--rhos 1` = no rescaling. For the paper's Figure-2 learning-rate sensitivity, run the optional cell in section 6.)

In [9]:
FINAL_SPECS = "freon:0.6667:0.01:0.001 freon:0.75:0.01:0.0005 kaon:0.01:0.0005 muon:0.01:0.002 sgd:0.5:0.001 truncsgd:0.05:0.5:0.002"  # <- REPLACE with the best specs from stage 1
!{sys.executable} sweep_cifar.py scale --opts {FINAL_SPECS} --rhos 1 --seeds 42 43 44 --freon_impl {FREON_IMPL} --out {RESULTS}/cifar_final.jsonl

{'mode': 'scale', 'tag': 'freon:0.6667', 'arch': 'vit', 'matrix_lr': 0.01, 'base_lr': 0.001, 'seed': 42, 'rho': 1.0, 'c': 0.6667, 'final_test_loss': 0.655835107421875, 'final_test_acc': 0.849, 'best_test_acc': 0.8515, 'diverged': False}
{'mode': 'scale', 'tag': 'freon:0.6667', 'arch': 'vit', 'matrix_lr': 0.01, 'base_lr': 0.001, 'seed': 43, 'rho': 1.0, 'c': 0.6667, 'final_test_loss': 0.6479300903320313, 'final_test_acc': 0.8497, 'best_test_acc': 0.8497, 'diverged': False}
{'mode': 'scale', 'tag': 'freon:0.6667', 'arch': 'vit', 'matrix_lr': 0.01, 'base_lr': 0.001, 'seed': 44, 'rho': 1.0, 'c': 0.6667, 'final_test_loss': 0.6464157043457032, 'final_test_acc': 0.8486, 'best_test_acc': 0.8494, 'diverged': False}
{'mode': 'scale', 'tag': 'freon:0.75', 'arch': 'vit', 'matrix_lr': 0.01, 'base_lr': 0.0005, 'seed': 42, 'rho': 1.0, 'c': 0.75, 'final_test_loss': 0.6522783447265625, 'final_test_acc': 0.8454, 'best_test_acc': 0.8458, 'diverged': False}
{'mode': 'scale', 'tag': 'freon:0.75', 'arch': 'v

## 5. Summary: mean ± std final test accuracy over seeds

In [10]:
import json, collections, statistics as st
rows = [json.loads(l) for l in open(f"{RESULTS}/cifar_final.jsonl") if l.strip()]
by = collections.defaultdict(list)
for r in rows:
    by[r["tag"]].append(r)
print(f"{'optimizer':16s} {'test acc':>17s} {'test loss':>17s}  n  diverged")
for tag, rs in sorted(by.items(), key=lambda kv: -st.mean(r["final_test_acc"] for r in kv[1])):
    acc = [r["final_test_acc"] for r in rs]; loss = [r["final_test_loss"] for r in rs]
    sd = lambda v: st.stdev(v) if len(v) > 1 else float("nan")
    print(f"{tag:16s} {st.mean(acc):.4f} ± {sd(acc):.4f}  {st.mean(loss):.4f} ± {sd(loss):.4f}  {len(rs)}  "
          f"{sum(r['diverged'] for r in rs)}")

optimizer                 test acc         test loss  n  diverged
muon             0.8493 ± 0.0013  0.6939 ± 0.0040  3  0
freon:0.6667     0.8491 ± 0.0006  0.6501 ± 0.0051  3  0
freon:0.75       0.8429 ± 0.0024  0.6584 ± 0.0119  3  0
kaon             0.8429 ± 0.0042  0.6791 ± 0.0097  3  0
truncsgd:0.05    0.8121 ± 0.0017  0.6408 ± 0.0106  3  0
sgd              0.7926 ± 0.0090  0.6191 ± 0.0115  3  0


## 6. (Optional, if time) learning-rate sensitivity — paper Fig. 2
Scales each tuned pair by ρ ∈ {0.03, 0.1, 0.3, 1, 3, 10, 30} over 3 seeds (7× the stage-2 cost).
ρ = 1 rows are recomputed, not reused, since they go to a separate file.

In [ ]:
!{sys.executable} sweep_cifar.py scale --opts {FINAL_SPECS} --freon_impl {FREON_IMPL} --out {RESULTS}/cifar_scale.jsonl
!{sys.executable} analyze_cifar.py scale {RESULTS}/cifar_scale.jsonl
from IPython.display import Image; Image(f"{RESULTS}/cifar_scale.png")